In [1]:
!pip install --upgrade unsloth unsloth_zoo
# !pip install vllm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.3/52.3 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.7/311.7 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 184.8/184.8 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.9/511.9 kB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.2/117.2 MB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.6/213.6 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.2/129.2 kB 12.4 MB/s eta 0:00:00
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [1]:
import os, math, numpy as np
# os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
# os.environ["CUDA_VISIBLE_DEVICES"]="0,1"
# os.environ["VLLM_USE_V1"] = "0"

In [2]:
from huggingface_hub import login
from google.colab import userdata
HF_key = userdata.get('PLo_HF')
login(token = HF_key)
# model_save = 'awilliam60412/0824-Llama-3-2-3B-Instruct-16bit-2E'
# model_save = 'awilliam60412/0824-Llama-3-1-8B-Instruct-16bit-2E'
# model_save = 'awilliam60412/0824-Llama-3-2-1B-Instruct-16bit-3E'
# model_save = 'awilliam60412/0824-Llama-3-2-3B-Instruct-16bit-3E'
# model_save = 'awilliam60412/0824-Llama-3-1-8B-Instruct-16bit-3E'
# model_save = 'unsloth/Llama-3.2-1B-Instruct'
# model_save = 'unsloth/Llama-3.2-3B-Instruct'
# model_save = 'unsloth/Meta-Llama-3.1-8B-Instruct'
model_save = 'nvidia/NVIDIA-Nemotron-Nano-9B-v2'

In [3]:
import pandas as pd
from google.colab import drive
import torch
drive.mount('/content/drive', force_remount=True)
FOLDERNAME = 'Colab Notebooks'
%cd drive/MyDrive/$FOLDERNAME/
test = pd.read_csv(f"Peter/data/test_data.csv")
sub = test[['row_id']].copy()
# examples = pd.read_csv(f'Peter/data/few_shot_examples.csv')

Mounted at /content/drive
/content/drive/MyDrive/Colab Notebooks


In [4]:
import os
import math
import numpy as np
import pandas as pd
import torch
from unsloth import FastLanguageModel
from transformers import LogitsProcessor, LogitsProcessorList
from sklearn.metrics import roc_auc_score
from tqdm import tqdm

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [5]:
# --- 1. THE FIX: A TRANSFORMERS-COMPATIBLE LOGITS PROCESSOR ---
# Since Unsloth uses the standard Hugging Face `generate`, we use this style of processor.
class YesNoLogitsProcessor(LogitsProcessor):
    # ✅ FIX 1: The __init__ method must accept the 'model' to get its vocab size.
    def __init__(self, model, tokenizer):
        self.tokenizer = tokenizer
        self.no_token_id = tokenizer.encode("No", add_special_tokens=False)[0]
        self.yes_token_id = tokenizer.encode("Yes", add_special_tokens=False)[0]

        # ✅ FIX 2: Create the mask using model.config.vocab_size. This guarantees the shape will match.
        self.allowed_mask = torch.zeros(model.config.vocab_size, dtype=torch.bool)

        self.allowed_mask[self.no_token_id] = True
        self.allowed_mask[self.yes_token_id] = True
        print(f"Constraining to tokens: No ({self.no_token_id}), Yes ({self.yes_token_id})")
        print(f"Mask shape created with model's vocab size: {self.allowed_mask.shape[0]}")

    # ✅ FIX 3: You must include the __call__ method for the class to work.
    def __call__(self, input_ids: torch.LongTensor, scores: torch.FloatTensor) -> torch.FloatTensor:
        """This method is called by the generator to process logits."""
        mask = self.allowed_mask.to(scores.device)

        # ✅ THE FIX: Apply the 1D mask to the first row of the 2D scores tensor.
        scores[0, ~mask] = -float('inf')

        return scores

In [9]:
# --- 2. MODEL LOADING with UNSLOTH ---
max_seq_length = 3072
dtype = None # None for auto detection
load_in_4bit = False

print("Loading model with Unsloth...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_save,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    trust_remote_code=True, # Added to handle custom code in the model
)
print("Model and tokenizer loaded.")
FastLanguageModel.for_inference(model)

Loading model with Unsloth...
Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2025.8.9: Fast Nemotron patching. Transformers: 4.55.2.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.557 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 8.0. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Nemotron does not support SDPA - switching to eager!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

nvidia/NVIDIA-Nemotron-Nano-9B-v2 does not have a padding token! Will use pad_token = <SPECIAL_999>.
Model and tokenizer loaded.


NemotronHForCausalLM(
  (backbone): NemotronHModel(
    (embeddings): Embedding(131072, 4480)
    (layers): ModuleList(
      (0): NemotronHBlock(
        (norm): NemotronHRMSNorm()
        (mixer): NemotronHMamba2Mixer(
          (act): SiLU()
          (conv1d): Conv1d(12288, 12288, kernel_size=(4,), stride=(1,), padding=(3,), groups=12288)
          (in_proj): Linear(in_features=4480, out_features=22656, bias=False)
          (norm): MambaRMSNormGated()
          (out_proj): Linear(in_features=10240, out_features=4480, bias=False)
        )
      )
      (1): NemotronHBlock(
        (norm): NemotronHRMSNorm()
        (mixer): NemotronHMLP(
          (up_proj): Linear(in_features=4480, out_features=15680, bias=False)
          (down_proj): Linear(in_features=15680, out_features=4480, bias=False)
          (act_fn): ReLUSquaredActivation()
        )
      )
      (2): NemotronHBlock(
        (norm): NemotronHRMSNorm()
        (mixer): NemotronHMamba2Mixer(
          (act): SiLU()
    

In [10]:
def formatting(dataset):
    texts = []
    for i in range(len(dataset)):
        texts.append(tokenizer.apply_chat_template(dataset[i], tokenize=False, add_generation_prompt=False))
    return texts

In [11]:
sys_prompt = ""
dataset = []
all_prompts = []

In [12]:
# --- 3. PROMPT & DATA SETUP (Same as before) ---
# (The template and prompt formatting logic remains the same as your original code)
template = """Subreddit: r/{subreddit}
Rule: {rule}
Examples:
1) {positive_example_1}
Violation: Yes
2) {negative_example_1}
Violation: No
3) {negative_example_2}
Violation: No
4) {positive_example_2}
Violation: Yes
Comment:
{body}
Violation:"""

sys_prompt_template = """
Rule: {rule}
Example: {example}
Violation: {Label}
"""
# Reasoning: {reasoning}

sys_prompt = 'You are given a comment on reddit and a rule. Your task is to classify whether the comment violates the rule. Only respond Yes/No. No need to explain.'

# sys_prompt += "Here are some examples for your reference."
# for index,row in examples.iterrows():
#   if row.Label == 1:
#     llabel = "Yes"
#   else:
#     llabel = "No"
#   print(llabel)
#   few_shot = sys_prompt_template.format(rule = row.rule, example = row.example, Label = llabel)
#   # few_shot = sys_prompt_template.format(rule = row.rule, example = row.example, Label = row.Label, reasoning = row.reasoning)
#   sys_prompt += few_shot


dataset = []
for index,row in test.iterrows():

    formatted_sample = [
        {
        "role": "system",
        "content": sys_prompt
    },
       {
           "role": "user",
           "content": template.format(
               rule = row.rule,
               subreddit = row.subreddit,
               body = row.body,
               positive_example_1 = row.positive_example_1,
               negative_example_1 = row.negative_example_1,
               positive_example_2 = row.positive_example_2,
               negative_example_2 = row.negative_example_2
           )
       }]

    dataset.append( formatted_sample )
all_prompts = formatting(dataset)
# dataset = []
# for index, row in test.iterrows():
#     dataset.append([
#         {"role": "system", "content": sys_prompt},
#         {"role": "user", "content": template.format(**row.to_dict())},
#     ])
# all_prompts = [tokenizer.apply_chat_template(d, tokenize=False, add_generation_prompt=True) for d in dataset]

In [14]:
# --- 4. GENERATION AND PROBABILITY EXTRACTION ---
# Instantiate the logits processor
# ✅ FIX 4: Pass BOTH the model and tokenizer to the processor.
yes_no_processor = YesNoLogitsProcessor(model, tokenizer)
logits_processor_list = LogitsProcessorList([yes_no_processor])

print("Generating responses and calculating probabilities...")
# The generation part remains the same
responses = []
for prompt in tqdm(all_prompts):
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens=2,
        logits_processor=logits_processor_list,
        output_scores=True,
        return_dict_in_generate=True
    )
    responses.append(outputs)

Constraining to tokens: No (4753), Yes (16860)
Mask shape created with model's vocab size: 131072
Generating responses and calculating probabilities...


  0%|          | 0/100 [00:00<?, ?it/s]


AttributeError: property 'key_cache' of 'HybridMambaAttentionDynamicCache' object has no setter

In [17]:
# --- 5. PROCESS RESULTS & CALCULATE AUC ---
probs_for_yes = []
predictions = [] # New list to store 0s and 1s

print("\n--- Processing Results ---")
for i, response in enumerate(responses):
    # Get the generated text
    generated_text = tokenizer.decode(response.sequences[0, -1]).strip()

    # ✅ **THE CHANGE IS HERE**
    # Convert text to integer prediction
    if generated_text == "Yes":
        prediction = 1
    else:
        prediction = 0
    predictions.append(prediction)

    # Probability calculation remains the same for AUC score
    first_token_logits = response.scores[0][0]
    yes_logit = first_token_logits[yes_no_processor.yes_token_id].item()
    no_logit = first_token_logits[yes_no_processor.no_token_id].item()
    softmax_probs = torch.nn.functional.softmax(torch.tensor([no_logit, yes_logit]), dim=0)
    normalized_yes_prob = softmax_probs[1].item()
    probs_for_yes.append(normalized_yes_prob)

    # Updated print statement
    # print(f"Comment {i+1}: Generated Text: '{generated_text}' -> Prediction: {prediction}")


--- Processing Results ---


In [18]:
print(probs_for_yes)

[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.8112776279449463, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.4902355968952179, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.8991213440895081, 0.7681540846824646, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.7264255881309509, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.849638819694519, 1.0, 1.0, 1.0, 1.0, 1.0]


In [19]:
ans = pd.read_csv(f"Peter/data/Ans.csv")
count = 0
for i in range(len(ans)):
  if predictions[i] >=0.5:
    val_ans = 1
  else:
    val_ans = 0
  if ans["rule_violation"][i] == val_ans:
    count += 1
print(count/len(ans))

0.48


In [ ]:
predictions

[0,
 0,
 0,
 1,
 0,
 1,
 1,
 0,
 0,
 1,
 1,
 0,
 0,
 0,
 0,
 1,
 1,
 1,
 1,
 0,
 0,
 0,
 1,
 0,
 1,
 1,
 0,
 1,
 1,
 0,
 1,
 0,
 0,
 0,
 1,
 1,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 1,
 0,
 1,
 1,
 0,
 0,
 1,
 0,
 1,
 1,
 1,
 1,
 0,
 1,
 1,
 1,
 0,
 0,
 1,
 1,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 1,
 1,
 1,
 1,
 0,
 1,
 1,
 1,
 1,
 1,
 0,
 1,
 1]

In [ ]:
sub['rule_violation'] = predictions

In [ ]:
sub.to_csv('submission-1B-base.csv')

In [ ]:
prob = test[['row_id']].copy()

In [ ]:
prob['rule_violation'] = probs_for_yes

In [ ]:
prob.to_csv('prob_submission-1B-base.csv')

In [8]:
!pip install mamba-ssm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.8/113.8 kB 3.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached ninja-1.13.0-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (5.1 kB)
Using cached ninja-1.13.0-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (180 kB)
  Created wheel for mamba-ssm: filename=mamba_ssm-2.2.5-cp312-cp312-linux_x86_64.whl size=323289691 sha256=c203c06afc2f4fb089cb792db54754ee4bb0b15b900cf71206c7dc2f44f13fe0
  Stored in directory: /root/.cache/pip/wheels/21/55/c4/85b634055d6a9b599d27f5cbeacf353c6c532d8e2d8769960b
Successfully built mamba-ssm
